# XLAV — E3: prompt engineering cho text prompt (CLIPSeg, PhraseCut miniv)

Câu hỏi: **cách viết câu lệnh** có làm thay đổi chất lượng mask không? Mỗi query được chạy với
4 template cố định từ trước: `{phrase}`, `a photo of {phrase}`, `the {phrase}`, `{phrase} in the image`.
Ảnh, GT, ngưỡng 0,3 và metric giữ nguyên như E1; chỉ chuỗi text thay đổi.

1. Bật **GPU T4 x2** và **Internet** trong Settings của notebook Kaggle.
2. **Run All**. Kết quả ở `/kaggle/working/clipseg_e3_prompt_templates`, kèm file ZIP.

Báo cáo **mọi template**, cả tốt lẫn kém. `{phrase}` là gốc để so sánh từng cặp (ΔIoU, CI 95% bootstrap theo ảnh).
Không chọn template "tốt nhất" trên evaluation rồi coi là kết quả độc lập; ngưỡng bổ sung chỉ chọn trên calibration.
Chọn 100 query bằng seed 42 và chia calibration/evaluation theo image_id giống hệt E1.


## 1. Dependency

Giữ PyTorch/CUDA, NumPy, Pillow, scikit-learn và matplotlib sẵn có của Kaggle.
Cell này chỉ pin Transformers và gdown; không cài wheel PyTorch CPU.
Nếu kernel đã import Transformers phiên bản khác trước đó, restart session rồi chạy lại từ đầu.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers==4.55.4", "gdown==5.2.0",
])


## 2. Cấu hình

`MAX_QUERIES=100` theo kế hoạch; đặt `8` để kiểm tra nhanh. `BATCH_SIZE` là số prompt mỗi lần forward
(query × template trên cùng ảnh); giảm nếu thiếu VRAM. `DATA_ROOT=None` tự tìm trong `/kaggle/input`
rồi tải từ nguồn chính thức nếu chưa có.


In [ ]:
from pathlib import Path

MAX_QUERIES = 100                # kế hoạch E3: 100 query; 8 = smoke
TEMPLATES = ["{phrase}", "a photo of {phrase}", "the {phrase}", "{phrase} in the image"]
BASELINE_TEMPLATE = "{phrase}"
BATCH_SIZE = 16                  # số prompt mỗi forward
OUTPUT_NAME = "clipseg_e3_prompt_templates"

DATA_ROOT = None                 # ví dụ: "/kaggle/input/phrasecut-miniv/VGPhraseCut_v0"
MODEL_PATH = None                # thư mục checkpoint offline, hoặc None
WORK_ROOT = Path("/kaggle/working")
REQUIRE_GPU = True               # False chỉ khi kiểm tra trên máy CPU
USE_AMP = False                  # mặc định FP32 như E1
SEED = 42
THRESHOLD = 0.3
CALIBRATION_FRACTION = 0.2
THRESHOLD_GRID = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
N_BOOTSTRAP = 2000               # bootstrap theo ảnh cho CI 95% của ΔIoU
TIE_TOLERANCE = 0.01             # |ΔIoU| ≤ 0,01 tính là hòa
SAVE_MASKS = True
N_VISUALIZATIONS = 6
DOWNLOAD_WORKERS = 4

MODEL_ID = "CIDAS/clipseg-rd64-refined"
MODEL_REVISION = "999e0328d9e10b484360c477313983f9afdd7050"
PHRASECUT_API_COMMIT = "276470b7773d8a230aee5b5536876fbc485fb04b"


In [ ]:
import csv
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import re
import shutil
import time
import zipfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from contextlib import nullcontext

import gdown
import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm
from transformers import CLIPSegForImageSegmentation, CLIPSegProcessor
from IPython.display import display

assert 0 < CALIBRATION_FRACTION < 1
assert 0 <= THRESHOLD <= 1 and all(0 <= t <= 1 for t in THRESHOLD_GRID)
assert N_BOOTSTRAP >= 100 and TIE_TOLERANCE >= 0
random.seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("Chưa có CUDA. Bật GPU trong Kaggle Settings rồi restart session.")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    try:
        torch.ones(1, device=device).add_(1).item()
    except RuntimeError as exc:
        raise RuntimeError("PyTorch của Kaggle không chạy được trên GPU này (thường gặp với P100). "
                           "Chọn Accelerator GPU T4 x2 rồi restart session.") from exc
assert not USE_AMP or device.type == "cuda", "FP16 AMP yêu cầu CUDA."
WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUT = WORK_ROOT / OUTPUT_NAME
OUT.mkdir(parents=True, exist_ok=True)
# Xóa kết quả lượt cũ để không trộn artifact vào ZIP.
for folder in ["masks", "visualizations"]:
    shutil.rmtree(OUT / folder, ignore_errors=True)
for filename in ["summary.json", "run.json"]:
    (OUT / filename).unlink(missing_ok=True)
(WORK_ROOT / f"{OUTPUT_NAME}.zip").unlink(missing_ok=True)

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")

def write_csv(path, rows):
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

versions = {name: importlib.metadata.version(name) for name in
            ["torch", "transformers", "numpy", "Pillow", "scikit-learn", "matplotlib", "gdown"]}
print("Device:", device, "| GPU:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")
print(json.dumps(versions, indent=2))


## 3. Lấy annotation chính thức và chọn query

Chỉ tải `refer_miniv.json` và metadata ảnh từ Drive ID trong
[downloader tác giả](https://github.com/ChenyunWu/PhraseCutDataset/blob/276470b7773d8a230aee5b5536876fbc485fb04b/download_dataset.py).
Chỉ các ảnh miniv được chọn mới được tải, không tải toàn bộ Visual Genome.
Nếu Google Drive hết quota, Add Input dữ liệu đã tải rồi đặt `DATA_ROOT`.

Chia calibration/evaluation theo **image_id**, seed 42, trước khi đọc prediction.
Nếu giới hạn số query, sampling bằng seed, lưu đúng danh sách vào `selected_tasks.json`.


In [ ]:
def metadata_path(root):
    for name in ["image_data_split.json", "image_data_split3000.json"]:
        candidate = root / name
        if candidate.is_file():
            return candidate
    return root / "image_data_split.json"

if DATA_ROOT is None:
    candidates = sorted(Path("/kaggle/input").rglob("refer_miniv.json")) if Path("/kaggle/input").exists() else []
    candidates = [p.parent for p in candidates if metadata_path(p.parent).is_file()]
    if len(candidates) > 1:
        raise ValueError("Có nhiều dataset miniv; đặt DATA_ROOT rõ ràng: " + str(candidates))
    data_root = candidates[0] if candidates else WORK_ROOT / "phrasecut_miniv"
else:
    data_root = Path(DATA_ROOT)
    if not data_root.is_dir():
        raise FileNotFoundError(f"DATA_ROOT không tồn tại: {data_root}")
data_root.mkdir(parents=True, exist_ok=True)
annotation_path = data_root / "refer_miniv.json"
info_path = metadata_path(data_root)

def ensure_json(path, file_id):
    if path.is_file():
        value = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(value, list) or not value:
            raise ValueError(f"JSON không phải danh sách annotation: {path}")
        return value
    if str(path).startswith("/kaggle/input/"):
        raise FileNotFoundError(f"Thiếu {path.name} trong input read-only; bổ sung Kaggle Dataset.")
    temp = path.with_suffix(".download")
    try:
        result = gdown.download(id=file_id, output=str(temp), quiet=False)
        if result is None:
            raise RuntimeError("Google Drive không trả về file.")
        value = json.loads(temp.read_text(encoding="utf-8"))
        if not isinstance(value, list) or not value:
            raise ValueError("Nội dung download không phải annotation JSON.")
        temp.replace(path)
        return value
    except Exception as exc:
        raise RuntimeError(f"Không tải được {path.name}. Bật Internet, hoặc Add Input dataset và đặt DATA_ROOT.") from exc

tasks = ensure_json(annotation_path, "1oLcTQ1blTIQyu5ZMelQN2HSniuQaxM4E")
image_info = ensure_json(info_path, "1xB9eExJo35K3DQu8PnBWZ1q_OMoMl8-i")
info_by_id = {int(row["image_id"]): row for row in image_info if row["split"] == "miniv"}
assert len({str(t["task_id"]) for t in tasks}) == len(tasks), "Task ID bị trùng."
assert all(int(t["image_id"]) in info_by_id for t in tasks), "Annotation không thuộc split miniv."
assert all(t.get("Polygons") and t.get("phrase", "").strip() for t in tasks), "Thiếu polygon GT hoặc phrase."
assert all(re.fullmatch(r"[A-Za-z0-9_-]+", str(t["task_id"])) for t in tasks), "Task ID không hợp lệ."

selected = sorted(tasks, key=lambda t: str(t["task_id"]))
if MAX_QUERIES is not None and MAX_QUERIES < len(selected):
    selected = random.Random(SEED).sample(selected, MAX_QUERIES)
selected.sort(key=lambda t: (int(t["image_id"]), str(t["task_id"])))
image_ids = sorted({int(t["image_id"]) for t in selected})
if len(image_ids) < 2:
    raise ValueError("Cần ít nhất 2 image_id để chia calibration/evaluation; tăng MAX_QUERIES.")
shuffled_ids = image_ids.copy()
random.Random(SEED).shuffle(shuffled_ids)
n_cal = min(len(image_ids) - 1, max(1, round(len(image_ids) * CALIBRATION_FRACTION)))
calibration_ids = set(shuffled_ids[:n_cal])
group_for_image = {i: "calibration" if i in calibration_ids else "evaluation" for i in image_ids}
assert not calibration_ids.intersection(set(image_ids) - calibration_ids)
write_json(OUT / "selected_tasks.json", [{"task_id": str(t["task_id"]), "image_id": int(t["image_id"]),
    "phrase": t["phrase"], "group": group_for_image[int(t["image_id"])]} for t in selected])
print(f"Annotation: {len(tasks)} queries / {len(info_by_id)} miniv images")
print(f"Selected: {len(selected)} queries / {len(image_ids)} images; calibration {n_cal} images")


## 4. Tải và kiểm tra ảnh

Ưu tiên ảnh đã có trong dataset input. Ảnh thiếu sẽ tải từ URL Visual Genome trong metadata
bằng HTTPS vào working cache, retry có giới hạn. Lỗi tải hoặc kích thước khác metadata làm
dừng lượt chạy: không âm thầm bỏ query để tạo số liệu đẹp hơn.


In [ ]:
image_cache = WORK_ROOT / "phrasecut_image_cache"
image_cache.mkdir(parents=True, exist_ok=True)
local_images = {}
for suffix in ["*.jpg", "*.jpeg", "*.png"]:
    for path in sorted(data_root.rglob(suffix)):
        if path.stem.isdigit() and int(path.stem) in image_ids:
            local_images.setdefault(int(path.stem), path)

def valid_image(path, info):
    with Image.open(path) as image:
        image.load()
        if image.size != (int(info["width"]), int(info["height"])):
            raise ValueError(f"{path.name}: ảnh {image.size} != metadata {(info['width'], info['height'])}")

def prepare_image(image_id):
    info = info_by_id[image_id]
    path = local_images.get(image_id, image_cache / f"{image_id}.jpg")
    if path.is_file():
        valid_image(path, info)
        return image_id, path
    url = info["url"].replace("http://", "https://", 1)
    temp = path.with_suffix(".download")
    for attempt in range(3):
        try:
            with requests.get(url, stream=True, timeout=(10, 60)) as response:
                response.raise_for_status()
                with temp.open("wb") as f:
                    for chunk in response.iter_content(1024 * 1024):
                        f.write(chunk)
            valid_image(temp, info)
            temp.replace(path)
            return image_id, path
        except Exception:
            if attempt == 2:
                raise
            time.sleep(attempt + 1)

image_paths, image_errors = {}, []
def prepare_safe(image_id):
    try:
        return prepare_image(image_id), None
    except Exception as exc:
        return None, {"image_id": image_id, "error": str(exc)}

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
    for result, error in tqdm(pool.map(prepare_safe, image_ids), total=len(image_ids), desc="Images"):
        if error:
            image_errors.append(error)
        else:
            image_paths[result[0]] = result[1]
write_json(OUT / "image_errors.json", image_errors)
if image_errors:
    raise RuntimeError(f"{len(image_errors)} ảnh lỗi; xem image_errors.json. Sửa dữ liệu rồi chạy lại.")
print("Validated images:", len(image_paths))


## 5. Nạp model, inference và metric

Model, rasterize GT và metric giống hệt E1. `predict` nhận **hoặc** danh sách text prompt
**hoặc** danh sách ảnh support (truyền vào `conditional_pixel_values` của CLIPSeg),
luôn dự đoán trên cùng một ảnh query. Logits nội suy bilinear về kích thước gốc rồi sigmoid.

`paired_comparison` so từng query với biến thể gốc: ΔIoU = biến thể − gốc, CI 95% bằng
bootstrap theo **ảnh** (các query cùng ảnh không độc lập), kèm số query thắng/hòa/thua.


In [ ]:
load_kwargs = {"local_files_only": True} if MODEL_PATH else {"revision": MODEL_REVISION}
model_source = str(MODEL_PATH) if MODEL_PATH else MODEL_ID
processor = CLIPSegProcessor.from_pretrained(model_source, use_fast=False, **load_kwargs)
model = CLIPSegForImageSegmentation.from_pretrained(model_source, **load_kwargs).to(device).eval()
assert processor.image_processor.size == {"height": 352, "width": 352}, "Processor không có input 352 như baseline."
assert model.config.vision_config.patch_size == 16, "Baseline yêu cầu ViT-B/16."
print("Model:", model_source, "| parameters:", sum(p.numel() for p in model.parameters()))

def rasterize_gt(task, width, height):
    mask = Image.new("L", (width, height), 0)
    draw = ImageDraw.Draw(mask)
    for instance in task["Polygons"]:
        for polygon in instance:
            if len(polygon) >= 2:
                draw.polygon([(int(x), int(y)) for x, y in polygon], outline=1, fill=1)
    target = np.asarray(mask, dtype=bool)
    if not target.any():
        raise ValueError(f"GT rỗng: task {task['task_id']}")
    return target

def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize(device)

def binary_metrics(probability, target, threshold):
    if probability.shape != target.shape or not target.any():
        raise ValueError("Shape không khớp hoặc GT rỗng.")
    prediction = probability >= threshold
    intersection = int(np.logical_and(prediction, target).sum())
    union = int(np.logical_or(prediction, target).sum())
    return {"intersection": intersection, "union": union, "iou": intersection / union,
            "dice": 2 * intersection / int(prediction.sum() + target.sum())}

def summarize(rows):
    if not rows:
        raise ValueError("Không có query để tính metric.")
    result = {"n_queries": len(rows), "n_images": len({r["image_id"] for r in rows}),
        "mean_sample_iou": float(np.mean([r["iou"] for r in rows])),
        "cumulative_iou": sum(r["intersection"] for r in rows) / sum(r["union"] for r in rows),
        "mean_dice": float(np.mean([r["dice"] for r in rows]))}
    for cutoff in [0.3, 0.5, 0.7, 0.9]:
        result[f"precision_at_{cutoff}"] = float(np.mean([r["iou"] >= cutoff for r in rows]))
    if "pixel_ap" in rows[0]:
        result["mean_sample_pixel_ap"] = float(np.mean([r["pixel_ap"] for r in rows]))
    return result

@torch.inference_mode()
def predict(image, prompts=None, supports=None):
    """Return [N,H,W] probabilities for N text prompts or N support images on one query image."""
    if (prompts is None) == (supports is None):
        raise ValueError("Truyền text prompts hoặc support images, không cả hai.")
    count = len(prompts) if prompts is not None else len(supports)
    inputs = dict(processor(images=[image] * count, return_tensors="pt"))
    if prompts is not None:
        inputs.update(processor.tokenizer(text=list(prompts), padding=True, truncation=True, return_tensors="pt"))
    else:
        inputs["conditional_pixel_values"] = processor(images=list(supports), return_tensors="pt")["pixel_values"]
    inputs = {k: v.to(device) for k, v in inputs.items()}
    synchronize()
    started = time.perf_counter()
    context = torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()
    with context:
        logits = model(**inputs).logits
    if logits.ndim == 2:
        logits = logits.unsqueeze(0)
    probabilities = F.interpolate(logits.float().unsqueeze(1), size=(image.height, image.width),
        mode="bilinear", align_corners=False).squeeze(1).sigmoid()
    synchronize()
    seconds = time.perf_counter() - started
    values = probabilities.cpu().numpy()
    if not np.isfinite(values).all():
        raise ValueError("Prediction chứa NaN/Inf; thử USE_AMP=False.")
    return values, seconds

def evaluate(probability, target, base):
    """Metric tại THRESHOLD (kèm pixel AP) và các dòng quét ngưỡng cho calibration."""
    row = {**base, "threshold": THRESHOLD, **binary_metrics(probability, target, THRESHOLD),
           "pixel_ap": float(average_precision_score(target.ravel(), probability.ravel()))}
    sweep = [{**base, "threshold": t, **binary_metrics(probability, target, t)} for t in THRESHOLD_GRID]
    return row, sweep

def variant_report(rows, sweep_rows):
    """Ngưỡng cố định trên mọi query; ngưỡng bổ sung chọn chỉ trên calibration."""
    table = []
    for threshold in THRESHOLD_GRID:
        subset = [r for r in sweep_rows if r["group"] == "calibration" and r["threshold"] == threshold]
        table.append({"threshold": threshold, **summarize(subset)})
    best = max(table, key=lambda r: (r["mean_sample_iou"], -r["threshold"]))["threshold"]
    return {
        "all_fixed_threshold": summarize(rows),
        "calibration_fixed_threshold": summarize([r for r in rows if r["group"] == "calibration"]),
        "evaluation_fixed_threshold": summarize([r for r in rows if r["group"] == "evaluation"]),
        "calibration_selected_threshold": best,
        "evaluation_calibrated_threshold": summarize([r for r in sweep_rows
            if r["group"] == "evaluation" and r["threshold"] == best]),
    }, table

def paired_comparison(base_iou, other_iou, image_of, n_bootstrap, tie_tolerance, seed):
    """ΔIoU theo query (other − base), CI 95% bootstrap theo ảnh, số query thắng/hòa/thua."""
    query_ids = sorted(base_iou)
    if set(query_ids) != set(other_iou):
        raise ValueError("Hai biến thể phải có cùng tập query.")
    deltas = np.array([other_iou[q] - base_iou[q] for q in query_ids])
    groups = defaultdict(list)
    for index, query_id in enumerate(query_ids):
        groups[image_of[query_id]].append(index)
    clusters = [np.array(indices) for _, indices in sorted(groups.items())]
    rng = np.random.default_rng(seed)
    boot = [deltas[np.concatenate([clusters[i] for i in rng.integers(0, len(clusters), len(clusters))])].mean()
            for _ in range(n_bootstrap)]
    return {"n_queries": len(deltas), "delta_mean_iou": float(deltas.mean()),
            "ci95_low": float(np.percentile(boot, 2.5)), "ci95_high": float(np.percentile(boot, 97.5)),
            "wins": int((deltas > tie_tolerance).sum()), "ties": int((np.abs(deltas) <= tie_tolerance).sum()),
            "losses": int((deltas < -tie_tolerance).sum())}

def overlay(image, probability):
    rgb = np.asarray(image).astype(np.float32).copy()
    mask = probability >= THRESHOLD
    rgb[mask] = 0.55 * rgb[mask] + 0.45 * np.array([255, 60, 60])
    return rgb.astype(np.uint8)

def load_rgb(image_id):
    with Image.open(image_paths[image_id]) as raw:
        return raw.convert("RGB")

def save_mask(probability, variant, stem):
    if SAVE_MASKS:
        folder = OUT / "masks" / variant
        folder.mkdir(parents=True, exist_ok=True)
        Image.fromarray((probability >= THRESHOLD).astype(np.uint8) * 255).save(folder / f"{stem}.png")

def table_row(name, report, comparison):
    fixed = report["all_fixed_threshold"]
    return {"variant": name, "rows": fixed["n_queries"], "mean_iou": fixed["mean_sample_iou"],
        "cumulative_iou": fixed["cumulative_iou"], "mean_dice": fixed["mean_dice"],
        "mean_pixel_ap": fixed["mean_sample_pixel_ap"], "precision_at_0.5": fixed["precision_at_0.5"],
        "calibrated_threshold": report["calibration_selected_threshold"],
        "evaluation_iou_fixed": report["evaluation_fixed_threshold"]["mean_sample_iou"],
        "evaluation_iou_calibrated": report["evaluation_calibrated_threshold"]["mean_sample_iou"],
        **{key: (comparison or {}).get(key) for key in
           ["delta_mean_iou", "ci95_low", "ci95_high", "wins", "ties", "losses"]}}

def print_table(rows):
    columns = ["variant", "rows", "mean_iou", "cumulative_iou", "mean_pixel_ap", "delta_mean_iou", "ci95_low", "ci95_high", "wins", "losses"]
    print(" | ".join(columns))
    for row in rows:
        print(" | ".join(f"{row[c]:.4f}" if isinstance(row[c], float) else str(row[c]) for c in columns))

def run_metadata(extra):
    inference_seconds = sum(b["forward_resize_seconds"] for b in batch_times)
    prompts = sum(b["prompts"] for b in batch_times)
    return {**extra,
        "model_id": MODEL_ID, "requested_revision": MODEL_REVISION if not MODEL_PATH else None,
        "model_source": model_source, "resolved_commit": getattr(model.config, "_commit_hash", None),
        "phrasecut_api_commit": PHRASECUT_API_COMMIT,
        "annotation_sha256": sha256(annotation_path), "image_metadata_sha256": sha256(info_path),
        "seed": SEED, "threshold": THRESHOLD, "threshold_grid": THRESHOLD_GRID,
        "calibration_fraction": CALIBRATION_FRACTION, "n_bootstrap": N_BOOTSTRAP, "tie_tolerance": TIE_TOLERANCE,
        "precision": "fp16_autocast" if USE_AMP else "fp32", "device": str(device),
        "gpu_name": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
        "torch_cuda_version": torch.version.cuda, "python": platform.python_version(), "versions": versions,
        "image_processor": processor.image_processor.to_dict(),
        "forward_resize_seconds_total": inference_seconds,
        "forward_resize_ms_per_prompt": 1000 * inference_seconds / prompts,
        "wall_seconds_inference_metrics_saving": elapsed_all,
        "peak_allocated_gpu_mb": torch.cuda.max_memory_allocated(device) / 1024**2 if device.type == "cuda" else None,
        "timing_note": "Two warmup batches; forward+resize+sigmoid timed with CUDA sync; excludes preprocessing and metric/save.",
    }


## 6. Chạy 4 template trên mọi query

Mỗi ảnh gom (query × template) thành batch. Mask lưu tại `masks/<t0..t3>/<task_id>.png`
(bảng ánh xạ key → template ở `run.json`). Hình minh họa đặt 4 template cạnh nhau cho cùng query.


In [ ]:
assert BATCH_SIZE >= 1
assert all("{phrase}" in t for t in TEMPLATES) and len(set(TEMPLATES)) == len(TEMPLATES)
assert BASELINE_TEMPLATE in TEMPLATES
template_keys = {template: f"t{i}" for i, template in enumerate(TEMPLATES)}
(OUT / "visualizations").mkdir(exist_ok=True)

tasks_by_image = defaultdict(list)
for task in selected:
    tasks_by_image[int(task["image_id"])].append(task)
warmup_image = load_rgb(image_ids[0])
warmup_prompts = [TEMPLATES[0].format(phrase=t["phrase"]) for t in tasks_by_image[image_ids[0]]][:BATCH_SIZE]
for _ in range(2):
    predict(warmup_image, prompts=warmup_prompts)
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats(device)

def save_template_figure(image, target, task, probabilities, ious, index):
    fig, axes = plt.subplots(1, 2 + len(TEMPLATES), figsize=(4 * (2 + len(TEMPLATES)), 4))
    axes[0].imshow(image); axes[0].set_title("Ảnh")
    axes[1].imshow(target, cmap="gray", vmin=0, vmax=1); axes[1].set_title("Ground truth")
    for ax, template in zip(axes[2:], TEMPLATES):
        ax.imshow(overlay(image, probabilities[template]))
        ax.set_title(f"{template_keys[template]}: IoU={ious[template]:.3f}\n{template.format(phrase='…')}")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"{task['task_id']} — {task['phrase']}")
    fig.tight_layout()
    fig.savefig(OUT / "visualizations" / f"{index:02d}_{task['task_id']}.png", dpi=100, bbox_inches="tight")
    plt.close(fig)

rows, sweep_rows, batch_times = [], [], []
visual_count = 0
started_all = time.perf_counter()
for image_id in tqdm(image_ids, desc="E3 images"):
    image = load_rgb(image_id)
    current = tasks_by_image[image_id]
    targets = {str(t["task_id"]): rasterize_gt(t, image.width, image.height) for t in current}
    jobs = [(task, template) for task in current for template in TEMPLATES]
    kept = defaultdict(dict)      # xác suất của các query được vẽ hình
    ious = defaultdict(dict)
    for start in range(0, len(jobs), BATCH_SIZE):
        chunk = jobs[start:start + BATCH_SIZE]
        probabilities, seconds = predict(image, prompts=[tpl.format(phrase=t["phrase"]) for t, tpl in chunk])
        batch_times.append({"prompts": len(chunk), "forward_resize_seconds": seconds})
        for (task, template), probability in zip(chunk, probabilities):
            task_id = str(task["task_id"])
            base = {"task_id": task_id, "image_id": image_id, "phrase": task["phrase"],
                "variant": template, "variant_key": template_keys[template],
                "prompt": template.format(phrase=task["phrase"]), "group": group_for_image[image_id],
                "width": image.width, "height": image.height}
            row, sweep = evaluate(probability, targets[task_id], base)
            rows.append(row)
            sweep_rows.extend(sweep)
            save_mask(probability, template_keys[template], task_id)
            ious[task_id][template] = row["iou"]
            kept[task_id][template] = probability
    for task in current:
        task_id = str(task["task_id"])
        if visual_count < N_VISUALIZATIONS:
            save_template_figure(image, targets[task_id], task, kept[task_id], ious[task_id], visual_count)
            visual_count += 1
    write_csv(OUT / "sample_metrics.csv", rows)
elapsed_all = time.perf_counter() - started_all
assert len(rows) == len(selected) * len(TEMPLATES), "Chưa đánh giá đủ query × template."
print(f"Completed {len(selected)} queries × {len(TEMPLATES)} templates; wall time {elapsed_all:.1f}s")


## 7. Bảng E3

`e3_template_table.csv` là bảng chính: mỗi template một dòng, metric tại t=0,3 trên toàn bộ query đã chọn,
ΔIoU so với `{phrase}` kèm CI 95% và số query thắng/hòa/thua. CI chứa 0 nghĩa là chưa đủ bằng chứng
template đó khác `{phrase}`. Cột calibrated chỉ để tham khảo, ngưỡng được chọn trên calibration.


In [ ]:
image_of = {str(t["task_id"]): int(t["image_id"]) for t in selected}
reports, calibration_rows, table = {}, [], []
iou_by_template = {}
for template in TEMPLATES:
    report, calibration_table = variant_report([r for r in rows if r["variant"] == template],
                                               [r for r in sweep_rows if r["variant"] == template])
    reports[template] = report
    calibration_rows += [{"variant": template, **r} for r in calibration_table]
    iou_by_template[template] = {r["task_id"]: r["iou"] for r in rows if r["variant"] == template}
for template in TEMPLATES:
    comparison = None if template == BASELINE_TEMPLATE else paired_comparison(
        iou_by_template[BASELINE_TEMPLATE], iou_by_template[template], image_of, N_BOOTSTRAP, TIE_TOLERANCE, SEED)
    reports[template]["paired_vs_baseline"] = comparison
    table.append({"variant_key": template_keys[template], **table_row(template, reports[template], comparison)})
write_csv(OUT / "calibration_thresholds.csv", calibration_rows)
write_csv(OUT / "e3_template_table.csv", table)
summary = {"experiment": "E3_prompt_templates", "dataset": "VGPhraseCut", "split": "miniv",
    "n_queries": len(selected), "complete_official_miniv": len(selected) == len(tasks),
    "baseline_template": BASELINE_TEMPLATE, "templates": reports,
    "metric_note": "IoU averaged per query at t=0.3; ΔIoU paired per query; CI is image-level bootstrap."}
write_json(OUT / "summary.json", summary)
write_json(OUT / "run.json", run_metadata({"experiment": "E3_prompt_templates", "max_queries": MAX_QUERIES,
    "batch_size": BATCH_SIZE, "templates": template_keys, "baseline_template": BASELINE_TEMPLATE,
    "selected_tasks_sha256": sha256(OUT / "selected_tasks.json"),
    "calibration_image_ids": sorted(calibration_ids),
    "evaluation_image_ids": sorted(set(image_ids) - calibration_ids)}))
print_table(table)


## 8. Xem hình và đóng gói

Tải `clipseg_e3_prompt_templates.zip` trong Kaggle Output.


In [ ]:
for path in sorted((OUT / "visualizations").glob("*.png"))[:N_VISUALIZATIONS]:
    with Image.open(path) as image:
        display(image.copy())
archive = WORK_ROOT / f"{OUTPUT_NAME}.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as package:
    for path in sorted(OUT.rglob("*")):
        if path.is_file():
            package.write(path, arcname=path.relative_to(OUT).as_posix())
print("Result folder:", OUT)
print("Download:", archive)


## Nguồn và giới hạn

- [CLIPSeg / CVPR 2022](https://github.com/timojl/clipseg), [checkpoint HF](https://huggingface.co/CIDAS/clipseg-rd64-refined).
- [PhraseCutDataset](https://github.com/ChenyunWu/PhraseCutDataset): annotation, URL ảnh, split và quy ước polygon/mask.
- [CLIPSeg trong Transformers 4.55.4](https://huggingface.co/docs/transformers/v4.55.4/en/model_doc/clipseg).

Checkpoint pretrained, không train. PhraseCut liên quan tới dữ liệu huấn luyện CLIPSeg nên không gọi là zero-shot.
Google Drive/Visual Genome có thể gián đoạn; Add Input dataset chứa hai file JSON (và ảnh) là đường dự phòng.
